# Electricity load forecasting

Load analysis and regression for customer `MT_158` using the Electricity Load Diagrams 2011-2014 dataset.

Source: Artur Trindade (2015), [ElectricityLoadDiagrams20112014](https://doi.org/10.24432/C58C86), UCI Machine Learning Repository, CC BY 4.0.

Run from this notebook's directory with pandas, scikit-learn and matplotlib installed. Install missing packages with `%pip install pandas scikit-learn matplotlib`.

The source archive is cached locally. Quarter-hour readings are averaged into hourly load in kW.

This exploratory example deliberately includes leakage, an unseeded split and a mutable data file for discussion in the lecture. Use the packaged project for the reproducible workflow.


In [ ]:
from pathlib import Path
import urllib.request
import zipfile
import pickle
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

print('Working directory:', Path.cwd())

## Load data

In [ ]:
if not Path('electricity.csv').exists():
    if not Path('electricity.zip').exists():
        print('Downloading UCI source once...')
        urllib.request.urlretrieve(
            'https://archive.ics.uci.edu/static/public/321/electricityloaddiagrams20112014.zip',
            'electricity.zip'
        )
    with zipfile.ZipFile('electricity.zip') as archive:
        with archive.open('LD2011_2014.txt') as handle:
            data = pd.read_csv(handle, sep=';', decimal=',', usecols=lambda name: name in ['Unnamed: 0', 'MT_158'])
    data.columns = ['timestamp', 'load_kw']
    data['timestamp'] = pd.to_datetime(data['timestamp'])
    data = data.set_index('timestamp').resample('1h', closed='right', label='right').mean()
    data = data.loc['2014-01-01 01:00:00':'2015-01-01 00:00:00']
    data.to_csv('electricity.csv')

raw = pd.read_csv('electricity.csv', parse_dates=['timestamp'])
print(raw.shape)
raw.head()

In [ ]:
raw.iloc[:168].plot(x='timestamp', y='load_kw', figsize=(12, 3), title='A week of customer MT_158')
plt.ylabel('Average load (kW)')
plt.show()

## Features

In [ ]:
df = raw.copy().set_index('timestamp')
df['lag_1h'] = df['load_kw'].shift(1)
df['lag_24h'] = df['load_kw'].shift(24)
df['lag_168h'] = df['load_kw'].shift(168)
df['rolling_mean'] = df['load_kw'].rolling(3, center=True).mean()
df['hour'] = df.index.hour
df['weekday'] = df.index.dayofweek

# Include the previous 24 hourly readings for both models.
for i in range(2, 24):
    df[f'lag_{i}h'] = df['load_kw'].shift(i)
df = df.dropna()
cols = [f'lag_{i}h' for i in range(1, 25)] + ['lag_168h', 'rolling_mean', 'hour', 'weekday']
X = df[cols]
y = df['load_kw']
X.head().round(2)

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2)

results = []
preds = {}
print('Training rows:', len(y_train), '| evaluation rows:', len(y_test))

## Ridge regression

In [ ]:
t0 = time.perf_counter()
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
preds['Ridge'] = predictions
results.append({'Model': 'Ridge', 'MAE (kW)': mean_absolute_error(y_test, predictions),
                'RMSE (kW)': root_mean_squared_error(y_test, predictions),
                'Time (s)': time.perf_counter() - t0})
with open('ridge.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler, 'columns': cols}, f)
with open('model_final.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler, 'columns': cols}, f)
display(pd.DataFrame(results).round(3))

## Random forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

t0 = time.perf_counter()
model2 = RandomForestRegressor(n_estimators=40, max_depth=10, min_samples_leaf=3, n_jobs=1)
model2.fit(X_train, y_train)
predictions = model2.predict(X_test)
preds['Random forest'] = predictions
results.append({'Model': 'Random forest', 'MAE (kW)': mean_absolute_error(y_test, predictions),
                'RMSE (kW)': root_mean_squared_error(y_test, predictions),
                'Time (s)': time.perf_counter() - t0})
with open('random_forest.pkl', 'wb') as f:
    pickle.dump({'model': model2, 'scaler': scaler, 'columns': cols}, f)
pd.DataFrame(results).round(3)

## Compare results

In [ ]:
scores = pd.DataFrame(results).sort_values('MAE (kW)').reset_index(drop=True)
display(scores.round(3))
scores.to_csv('model_scores.csv', index=False, float_format='%.6f')

prediction_table = pd.DataFrame({'timestamp': y_test.index, 'actual_kw': y_test.to_numpy()})
for name, values in preds.items():
    prediction_table[name] = values
prediction_table = prediction_table.sort_values('timestamp')
prediction_table.to_csv('predictions.csv', index=False, float_format='%.8f')

fig, ax = plt.subplots(figsize=(9, 3.5))
bars = ax.barh(scores['Model'], scores['MAE (kW)'], color='#007f86')
ax.bar_label(bars, fmt='%.3f', padding=5)
ax.set_xlim(0, scores['MAE (kW)'].max() * 1.2)
ax.invert_yaxis()
ax.set_xlabel('Mean absolute error (kW)')
ax.set_title('Model comparison')
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
low, high = float(y_test.min()), float(y_test.max())
for ax, (name, values) in zip(axes.flat, preds.items()):
    ax.scatter(y_test, values, s=6, alpha=0.35, color='#007f86')
    ax.plot([low, high], [low, high], '--', color='grey', linewidth=1)
    ax.set_title(f'{name} | MAE {mean_absolute_error(y_test, values):.3f} kW')
    ax.set_xlabel('Observed load (kW)')
    ax.set_ylabel('Predicted load (kW)')
    ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
fig.savefig('results.png', dpi=150)
plt.show()

## Reload a saved model

The random-forest bundle contains the fitted estimator, scaler and feature column order.


In [ ]:
with open('random_forest.pkl', 'rb') as f:
    saved = pickle.load(f)
new_features = X.loc[y_test.index, saved['columns']].iloc[:5]
scaled = saved['scaler'].transform(new_features)
check = saved['model'].predict(scaled)
display(pd.DataFrame({'Observed (kW)': y_test.iloc[:5].to_numpy(),
                      'Reloaded forecast (kW)': check}).round(3))


## Local data update

In [ ]:
raw.loc[500:515, 'load_kw'] *= 1.2
raw.to_csv('electricity.csv', index=False)
raw.to_csv('electricity_fixed.csv', index=False)
print('Saved ridge.pkl, random_forest.pkl and model_final.pkl')

## Output files

- `ridge.pkl`, `random_forest.pkl`: fitted models with scaling information and feature columns.
- `model_final.pkl`: latest saved Ridge model bundle.
- `model_scores.csv`: MAE, RMSE and training time for both models.
- `predictions.csv`: evaluation timestamps, observed load and both model predictions.
- `model_comparison.png`, `results.png`: comparison and diagnostic plots.
- `electricity.csv`, `electricity_fixed.csv`: local hourly readings.
